# Socratic AI Tutor - Setup and Testing

This notebook helps you set up and test the Socratic AI tutor system.

## Prerequisites

1. Install Ollama or LM Studio
2. Download Qwen2.5 model
3. Install Python dependencies

## Installation Steps

In [2]:
# Set up path to import socratic module
import sys
from pathlib import Path

project_root = Path().resolve().parent
sys.path.insert(0, str(project_root / "src"))

print(f"Project root: {project_root}")

Project root: <path> Assignments\CS 370\eng-ai-agents\project


## Test LLM Connection

In [3]:
from socratic.config import settings
from socratic.utils.llm_client import llm_client

# Test connection
print(f"LLM Provider: {settings.llm_provider}")
print(f"Base URL: {settings.get_llm_base_url()}")
print(f"Model: {settings.model_name}")

# Test generation
try:
    response = llm_client.generate(
        prompt="What is machine learning? Answer in one sentence.",
        temperature=0.7,
        max_tokens=100
    )
    print(f"\nTest response: {response}")
    print("\n✅ LLM connection successful!")
except Exception as e:
    print(f"\n❌ Error connecting to LLM: {e}")

LLM Provider: ollama
Base URL: http://localhost:11434/v1
Model: qwen2.5:latest

Test response: Machine learning is a method of teaching computers to recognize patterns and make decisions or predictions without being explicitly programmed.

✅ LLM connection successful!


## Milestone 2: Data Ingestion

Let's test the data ingestion pipeline.

In [4]:
from socratic.ingestion import scrape_course_website

# Set data directory to project root
data_dir = project_root / "data"

# Test web scraping (start with depth=1 for testing)
print("Scraping course website...")
print(f"Data will be saved to: {data_dir}")

resources = scrape_course_website(
    base_url="https://pantelis.github.io/courses/ai/in-person.html",
    max_depth=1,  # Start shallow for testing
    save_dir=str(data_dir / "raw" / "web")
)

print(f"\nScraped {len(resources)} web pages")
for i, resource in enumerate(resources[:3]):
    print(f"\n{i+1}. {resource.title}")
    print(f"   URL: {resource.url}")
    print(f"   Content length: {len(resource.content)} chars")

Scraping course website...
Data will be saved to: ./data

Scraped 41 web pages

1. Introduction to Artificial Intelligence – Engineering AI Agents
   URL: https://pantelis.github.io/courses/ai/in-person.html
   Content length: 1940 chars

2. Back2classroom – Engineering AI Agents
   URL: https://pantelis.github.io/index.html
   Content length: 14072 chars

3. Foundations – Engineering AI Agents
   URL: https://pantelis.github.io/book/foundations/index.html
   Content length: 1089 chars


In [5]:
# Test PDF processing
from socratic.ingestion import process_pdf
from pathlib import Path

# Use project root data directory
pdf_dir = data_dir / "raw" / "pdfs"
pdf_dir.mkdir(parents=True, exist_ok=True)

# Process PDFs if they exist
pdf_files = list(pdf_dir.glob("*.pdf"))
if pdf_files:
    print(f"Found {len(pdf_files)} PDF files")
    for pdf_path in pdf_files[:2]:  # Process first 2 for testing
        try:
            resource = process_pdf(pdf_path)
            resources.append(resource)
            print(f"✅ Processed: {resource.title}")
        except Exception as e:
            print(f"❌ Error processing {pdf_path.name}: {e}")
else:
    print(f"No PDFs found in {pdf_dir}. Add some PDFs to test!")

No PDFs found in ./data/raw/pdfs. Add some PDFs to test!


## Milestone 3: Knowledge Graph Construction

In [6]:
from socratic.graph import KnowledgeGraphBuilder

# Build knowledge graph from collected resources
print("Building knowledge graph...")
print("This may take a few minutes depending on the number of resources.\n")

builder = KnowledgeGraphBuilder()

# Start with just a few resources for testing
test_resources = resources[:3]
kg = builder.build_from_resources(test_resources)

print(f"\n✅ Knowledge graph built:")
print(f"   - {len(kg.concepts)} concepts")
print(f"   - {len(kg.resources)} resources")
print(f"   - {len(kg.examples)} examples")
print(f"   - {len(kg.edges)} edges")

INFO:socratic.graph.builder:Building knowledge graph from 3 resources


Building knowledge graph...
This may take a few minutes depending on the number of resources.



INFO:httpx:HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"
ERROR:socratic.graph.builder:Error extracting concepts from web_6876859567941453882: Expecting value: line 1 column 1 (char 0)
INFO:httpx:HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"
ERROR:socratic.graph.builder:Error extracting examples from web_6876859567941453882: Expecting value: line 1 column 1 (char 0)
INFO:httpx:HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"
ERROR:socratic.graph.builder:Error extracting concepts from web_6117206406427263275: Expecting value: line 1 column 1 (char 0)
INFO:httpx:HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"
ERROR:socratic.graph.builder:Error extracting examples from web_6117206406427263275: Expecting value: line 1 column 1 (char 0)
INFO:httpx:HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"
ERROR:socratic.graph.builder:Error ex


✅ Knowledge graph built:
   - 0 concepts
   - 3 resources
   - 0 examples
   - 0 edges


In [7]:
# Inspect some concepts
print("\nSample Concepts:")
for i, (concept_id, concept) in enumerate(list(kg.concepts.items())[:5]):
    print(f"\n{i+1}. {concept.title} (Difficulty: {concept.difficulty})")
    if concept.definitions:
        print(f"   Definition: {concept.definitions[0][:150]}...")


Sample Concepts:


In [8]:
# Save knowledge graph to project root data directory
kg_path = data_dir / "graph" / "knowledge_graph.json"
kg_path.parent.mkdir(parents=True, exist_ok=True)

builder.save(kg_path)
print(f"✅ Knowledge graph saved to {kg_path}!")

INFO:socratic.graph.builder:Knowledge graph saved to ./data/graph/knowledge_graph.json


✅ Knowledge graph saved to ./data/graph/knowledge_graph.json!


## Milestone 4: Query and Generation

In [9]:
from socratic.query import QueryEngine

# Create query engine
query_engine = QueryEngine(kg)

# Test query
test_query = "What is machine learning?"
print(f"Query: {test_query}\n")

result = query_engine.answer_query(test_query)

print("Answer:")
print(result["answer"])
print(f"\nConcepts used: {result['concepts_used']}")
print(f"\nCitations:")
for citation in result["citations"]:
    print(f"  - {citation['title']}")

INFO:socratic.graph.storage:NetworkX graph built: 3 nodes, 0 edges
INFO:socratic.query.engine:Processing query: What is machine learning?


Query: What is machine learning?



INFO:httpx:HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"
ERROR:socratic.query.engine:Error identifying concepts: Expecting value: line 1 column 1 (char 0)


Answer:
I couldn't identify any relevant concepts for your question. Could you rephrase or provide more context?

Concepts used: []

Citations:


## Test Evaluation Queries

Test the three required evaluation queries from the project spec.

In [10]:
# Query 1: Attention mechanisms in transformers
query1 = "How do attention mechanisms work in transformers? Can you show me a Python implementation?"
result1 = query_engine.answer_query(query1)

print("Query 1:", query1)
print("\nAnswer:")
print(result1["answer"])
print(f"\nNodes used: {result1['concepts_used']}")

INFO:socratic.query.engine:Processing query: How do attention mechanisms work in transformers? Can you show me a Python implementation?
INFO:httpx:HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"
INFO:socratic.query.engine:Identified 2 relevant concepts for query
INFO:socratic.query.engine:Retrieved context: 0 concepts, 0 prerequisites, 0 resources, 0 examples
INFO:httpx:HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"


Query 1: How do attention mechanisms work in transformers? Can you show me a Python implementation?

Answer:
{
  "answer": "Sure! To understand attention mechanisms in transformers, we need to build up from some foundational concepts first.

1. **Understanding Basic Neural Networks and Recurrent Neural Networks (RNNs)**:
   - A basic neural network processes data point by point.
   - In contrast, RNNs process sequences of data, but they struggle with long-term dependencies due to the vanishing gradient problem [Resource: Understanding LSTM, 2018].

2. **Introduction to Transformers**:
   - Transformers introduced a new architecture that uses self-attention mechanisms for parallel processing, addressing both the limitations and benefits of RNNs.
   - Self-attention allows each position in the sequence to attend to all other positions [Resource: Attention Is All You Need, 2017].

3. **Attention Mechanisms**:
   - The core idea is that a value can be attended by multiple keys. This mechan

In [11]:
# Query 2: CLIP architecture
query2 = "What is the CLIP architecture and how is it used in computer vision?"
result2 = query_engine.answer_query(query2)

print("Query 2:", query2)
print("\nAnswer:")
print(result2["answer"])
print(f"\nNodes used: {result2['concepts_used']}")

INFO:socratic.query.engine:Processing query: What is the CLIP architecture and how is it used in computer vision?
INFO:httpx:HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"
ERROR:socratic.query.engine:Error identifying concepts: Expecting value: line 1 column 1 (char 0)


Query 2: What is the CLIP architecture and how is it used in computer vision?

Answer:
I couldn't identify any relevant concepts for your question. Could you rephrase or provide more context?

Nodes used: []


In [12]:
# Query 3: Variational lower bound
query3 = "Explain the variational lower bound and its connection to Jensen's inequality"
result3 = query_engine.answer_query(query3)

print("Query 3:", query3)
print("\nAnswer:")
print(result3["answer"])
print(f"\nNodes used: {result3['concepts_used']}")

INFO:socratic.query.engine:Processing query: Explain the variational lower bound and its connection to Jensen's inequality
INFO:httpx:HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"
INFO:socratic.query.engine:Identified 2 relevant concepts for query
INFO:socratic.query.engine:Retrieved context: 0 concepts, 0 prerequisites, 0 resources, 0 examples
INFO:httpx:HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"


Query 3: Explain the variational lower bound and its connection to Jensen's inequality

Answer:
{
  "answer": {
    "content": "Certainly! Let's break down the variational lower bound and its connection to Jensen's inequality in a step-by-step manner."

    // Step 1: Explain Jensen's Inequality
    "jensen_explanation": "First, let's explore Jensen's Inequality. This inequality describes how a convex function behaves when applied to the expected value of a random variable. For a convex function \( f \) and any random variable \( X \), we have:

    \\[
    f(\mathbb{E}[X]) \leq \mathbb{E}[f(X)]
    \\]

    This means that if you take an average (expected value) of the function applied to the random variable, it will be at least as large as applying the function to the average first. For a concave function \( f \), the inequality is reversed.

    Example: Consider the convex function \( f(x) = x^2 \). If we take a random variable \( X \) with values 1 and -1, then:

    \\[
    \math

## Visualize Knowledge Graph

In [13]:
from socratic.graph import GraphStorage

# Create visualization
storage = GraphStorage(kg)

# Get first few concept IDs
concept_ids = list(kg.concepts.keys())[:5]

# Save to project root data directory
output_path = data_dir / "graph" / "subgraph_viz.png"
output_path.parent.mkdir(parents=True, exist_ok=True)

storage.visualize_subgraph(concept_ids, str(output_path))
print(f"✅ Visualization saved to {output_path}")

INFO:socratic.graph.storage:NetworkX graph built: 3 nodes, 0 edges
INFO:socratic.graph.storage:Extracted subgraph with 0 nodes
./src/socratic/graph/storage.py:208: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
INFO:socratic.graph.storage:Subgraph visualization saved to ./data/graph/subgraph_viz.png


✅ Visualization saved to ./data/graph/subgraph_viz.png


## Next Steps

1. **M2**: Run full data ingestion with more URLs and PDFs
2. **M3**: Build complete knowledge graph from all resources
3. **M4**: Test and refine query responses
4. **Launch Chat Interface**: Use `python -m socratic.main` to start interactive chat

See README.md for more details!